In [ ]:
# [0 · Imports & configuration]
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

sc.set_figure_params(dpi=100, frameon=False)
plt.rcParams['figure.max_open_warning'] = 0

print(f'scvi-tools: {scvi.__version__}')

from b_cell_utils import *

CACHE_DIR        = Path('/home/projects/nyosef/zvise/PixelGen/PixelGen/New_Data/cache')
ANNOTATED_CACHE  = CACHE_DIR / 'adata_cytovi_annotated_compat.h5ad'
ASINH_SCALE      = 5.0
FDR_THRESH       = 0.05

In [ ]:
# [1 · Data loading & subsetting for B cells]
adata = sc.read_h5ad(ANNOTATED_CACHE)

# Subset to 6h Mock, B cells only, comparing two cell systems
SYS_A = 'NALM-6 + healthy T'
SYS_B = 'healthy B + healthy T'

mask_6h_mock_b = (
    (adata.obs['time'] == '6h') &
    (adata.obs['condition'] == 'Mock') &
    (adata.obs['cell_type_annot'] == 'B') &
    (adata.obs['cell_system'].isin([SYS_A, SYS_B]))
)

adata_6h = adata[mask_6h_mock_b].copy()

print(f'6h Mock B cells (both systems): {adata_6h.n_obs}')
print(f'\nCell counts by sample × system × type:')
print(pd.crosstab(
    index=adata_6h.obs['sample'],
    columns=[adata_6h.obs['cell_system']],
    margins=True
))

print(f'\nMarkers: {adata_6h.n_vars}')
print(f'\nData layers: {list(adata_6h.layers.keys())}')
print(f'Spatial representations (obsm): {list(adata_6h.obsm.keys())}')

In [ ]:
sc.pl.umap(adata_6h, color=['cell_system','condition'], frameon=False)

In [ ]:
# [2 · UMAP: B cell marker subtypes]
import json

# Load marker panel
with open('marker_panels.json') as f:
    marker_panels = json.load(f)

b_cell_subtypes = marker_panels['b_cell_markers']

print(f'B cell marker subtypes: {list(b_cell_subtypes.keys())}')
print(f'\nTotal B cell markers: {sum(len(v) for v in b_cell_subtypes.values())}')

# Create one figure per subtype
for subtype_name, markers in b_cell_subtypes.items():
    # Filter to markers present in data
    markers_present = [m for m in markers if m in adata_6h.var_names]
    print(f'\n{subtype_name}: {len(markers_present)}/{len(markers)} present')
    print(f'  Markers: {markers_present}')

    if not markers_present:
        print(f'  (skipping — no markers found)')
        continue

    # UMAP with proper individual titles per marker
    plot_umap_markers(adata_6h, markers_present, title_prefix=f'B cells — {subtype_name}')

In [ ]:
# [2a · CD19 histogram on CD8 T cells — NALM-6 + healthy T]
mask_cd8_nalm = (
    (adata.obs['cell_system'] == 'NALM-6 + healthy T') &
    (adata.obs['cell_type_annot'] == 'CD8 T')
)
adata_cd8_nalm = adata[mask_cd8_nalm]

cd19_vals = np.array(adata_cd8_nalm[:, 'CD19'].layers['arcsinh']).flatten()

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(cd19_vals, bins=50, color='#d62728', alpha=0.8, edgecolor='white', linewidth=0.5)
ax.set_xlabel('CD19 (arcsinh)', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title(f'CD19 expression on CD8 T cells — NALM-6 + healthy T\n(n={mask_cd8_nalm.sum()})',
             fontsize=12, fontweight='bold')
ax.axvline(cd19_vals.mean(), color='black', linestyle='--', linewidth=1,
           label=f'mean = {cd19_vals.mean():.2f}')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# [2b · CD19 histogram on B cells — NALM-6 vs Healthy]
mask_b_nalm = (
    (adata.obs['cell_system'] == 'NALM-6 + healthy T') &
    (adata.obs['cell_type_annot'] == 'B')
)
mask_b_healthy = (
    (adata.obs['cell_system'] == 'healthy B + healthy T') &
    (adata.obs['cell_type_annot'] == 'B')
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, layer, label in zip(axes, ['raw', 'arcsinh'], ['Raw counts', 'arcsinh']):
    vals_nalm = np.array(adata[mask_b_nalm, 'CD19'].layers[layer]).flatten()
    vals_healthy = np.array(adata[mask_b_healthy, 'CD19'].layers[layer]).flatten()

    ax.hist(vals_healthy, bins=50, color='#1f77b4', alpha=0.6, edgecolor='white', linewidth=0.5,
            label=f'Healthy B (n={len(vals_healthy)}, mean={vals_healthy.mean():.2f})')
    ax.hist(vals_nalm, bins=50, color='#d62728', alpha=0.6, edgecolor='white', linewidth=0.5,
            label=f'NALM-6 B (n={len(vals_nalm)}, mean={vals_nalm.mean():.2f})')
    ax.axvline(vals_healthy.mean(), color='#1f77b4', linestyle='--', linewidth=1)
    ax.axvline(vals_nalm.mean(), color='#d62728', linestyle='--', linewidth=1)
    ax.set_xlabel(f'CD19 ({label})', fontsize=11)
    ax.set_ylabel('Count', fontsize=11)

    _, pv = mannwhitneyu(vals_nalm, vals_healthy, alternative='two-sided')
    ax.set_title(f'CD19 ({label}) — NALM-6 vs Healthy B cells\nMW p = {pv:.2e}  {sig_label(pv)}',
                 fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# [2c · Dotplot: B cell differentiation stage markers — NALM-6 vs Healthy]
# Key markers for staging B cell differentiation from available panel
DIFF_MARKERS = {
    'Progenitor / Pre-B':      [ 'CD117', 'CD10', 'CD127'],
    'Pan-B / Identity':        ['CD19',  'CD20', 'CD22'],
    'Immature / Transitional': ['CD24', 'CD38', 'IgM'],
    'Mature / Naive':          ['CD21', 'CD268', 'IgD'],
    'Memory / Activated':      ['CD27', 'CD80', 'CD86', 'CD95', 'CD40'],
    'Plasmablast / Plasma':    ['CD138', 'CD269'],
}

# Filter to present markers, keep order
markers_ordered = []
var_group_labels = []
var_group_positions = []
pos = 0
for stage, markers in DIFF_MARKERS.items():
    present = [m for m in markers if m in adata.var_names]
    if present:
        var_group_labels.append(stage)
        var_group_positions.append((pos, pos + len(present) - 1))
        markers_ordered.extend(present)
        pos += len(present)

print(f'Differentiation markers available: {len(markers_ordered)}')
for stage, markers in DIFF_MARKERS.items():
    present = [m for m in markers if m in adata.var_names]
    missing = [m for m in markers if m not in adata.var_names]
    print(f'  {stage}: {[display_name(m) for m in present]}', end='')
    if missing:
        print(f'  (missing: {missing})', end='')
    print()

# Subset B cells from both systems (all timepoints)
mask_b_both = (
    (adata.obs['cell_type_annot'] == 'B') &
    (adata.obs['cell_system'].isin([SYS_A, SYS_B]))
)
adata_b = adata[mask_b_both].copy()
adata_b.obs['system'] = adata_b.obs['cell_system'].map({
    SYS_A: 'NALM-6',
    SYS_B: 'Healthy B',
}).astype('category')

sc.pl.dotplot(
    adata_b, var_names=markers_ordered, groupby='system',
    layer='arcsinh',
    var_group_labels=[s for s in var_group_labels],
    var_group_positions=var_group_positions,
    figsize=(len(markers_ordered) * 0.55, 3),
)

In [ ]:
# [2d · Leiden clustering of B cells + dotplot by cluster × sample]
# PCA + neighbors + leiden on arcsinh layer
adata_b.X = np.array(adata_b.layers['arcsinh'], dtype=np.float32)
sc.pp.pca(adata_b, n_comps=20)
sc.pp.neighbors(adata_b, n_neighbors=15, n_pcs=20)
sc.tl.umap(adata_b)
sc.tl.leiden(adata_b, resolution=0.3, key_added='leiden')

print(f'Leiden clusters (res=0.3): {adata_b.obs["leiden"].nunique()}')
print(adata_b.obs.groupby(['leiden', 'system']).size().unstack(fill_value=0))

sc.pl.umap(adata_b, color=['leiden', 'system'], frameon=False, ncols=2)

In [ ]:
# [2e · Dotplot: differentiation markers by leiden cluster × sample]
# Combined group: system first so rows cluster by origin
adata_b.obs['sample_leiden'] = (
    adata_b.obs['system'].astype(str) + ' · C' +
    adata_b.obs['leiden'].astype(str)
)

# Sort so NALM-6 clusters come first, then Healthy B
cat_order = sorted(adata_b.obs['sample_leiden'].unique(),
                   key=lambda x: (0 if x.startswith('NALM') else 1, x))
adata_b.obs['sample_leiden'] = pd.Categorical(
    adata_b.obs['sample_leiden'], categories=cat_order, ordered=True
)

sc.pl.dotplot(
    adata_b, var_names=markers_ordered, groupby='sample_leiden',
    layer='arcsinh',
    var_group_labels=var_group_labels,
    var_group_positions=var_group_positions,
    figsize=(len(markers_ordered) * 0.55, len(cat_order) * 0.45),
)

# Also show per-cluster composition
comp = pd.crosstab(adata_b.obs['leiden'], adata_b.obs['system'], margins=True)
print(f"\n{'='*50}")
print(f"  Cluster composition")
print(f"{'='*50}")
print(comp)

In [ ]:
# [3 · Differential abundance (arcsinh)]
plot_da_bars(adata_6h, ['B'], SYS_A, SYS_B)

In [ ]:
# [4 · Subset: healthy B+T, B cells, all timepoints & conditions]
mask_healthy_b = (
    (adata.obs['cell_system'] == 'healthy B + healthy T') &
    (adata.obs['cell_type_annot'] == 'B')
)
adata_healthy_b = adata[mask_healthy_b].copy()

# Create combined group label
adata_healthy_b.obs['time_cond'] = (
    adata_healthy_b.obs['time'].astype(str) + ' ' +
    adata_healthy_b.obs['condition'].astype(str)
)

print(f'B cells in healthy B+T: {adata_healthy_b.n_obs}')
print(adata_healthy_b.obs['time_cond'].value_counts().to_string())

In [ ]:
# [5 · 4-way DA: B cells healthy B+T vs NALM-6+T (6h/48h × Mock/Blina)]
B_CELL_MARKERS_SET = set(load_marker_panel('b_cell_markers'))

# Comparison: B cells in NALM-6 + healthy T
mask_nalm_b = (
    (adata.obs['cell_system'] == 'NALM-6 + healthy T') &
    (adata.obs['cell_type_annot'] == 'B')
)
adata_nalm_b = adata[mask_nalm_b].copy()
adata_nalm_b.obs['time_cond'] = (
    adata_nalm_b.obs['time'].astype(str) + ' ' +
    adata_nalm_b.obs['condition'].astype(str)
)

print(f'B cells healthy B+T: {adata_healthy_b.n_obs}')
print(f'B cells NALM-6+T:    {adata_nalm_b.n_obs}')

plot_marker_panel_violins(
    adata_healthy_b, 'b_cell_markers',
    group_key='time_cond',
    adata_compare=adata_nalm_b,
    primary_label='Healthy B + T',
    compare_label='NALM-6 + T',
)

In [ ]:
# [7 · Spatial subsets for selected-marker comparisons]
# Build spatial colocalization DataFrames for B cells in each condition × system

def _sp_subset_b(adata_full, time_val, cond_val, system_val):
    """Return spatial_asinh5 DataFrame for B cells matching filters."""
    mask = (
        (adata_full.obs['time'] == time_val) &
        (adata_full.obs['condition'] == cond_val) &
        (adata_full.obs['cell_type_annot'] == 'B') &
        (adata_full.obs['cell_system'] == system_val)
    )
    sp = adata_full[mask].obsm['spatial_asinh5']
    if not isinstance(sp, pd.DataFrame):
        sp = pd.DataFrame(sp, index=adata_full.obs_names[mask])
    print(f'  {time_val} {cond_val:15s} {system_val:25s} → {sp.shape[0]} B cells')
    return sp

SYS_H = 'healthy B + healthy T'
SYS_N = 'NALM-6 + healthy T'

print('Building spatial subsets (B cells only):')
sp_6h_mock_h  = _sp_subset_b(adata, '6h',  'Mock',          SYS_H)
sp_6h_mock_n  = _sp_subset_b(adata, '6h',  'Mock',          SYS_N)
sp_6h_blina_h = _sp_subset_b(adata, '6h',  'Blinatumomab',  SYS_H)
sp_6h_blina_n = _sp_subset_b(adata, '6h',  'Blinatumomab',  SYS_N)
sp_48h_blina_h = _sp_subset_b(adata, '48h', 'Blinatumomab', SYS_H)
sp_48h_blina_n = _sp_subset_b(adata, '48h', 'Blinatumomab', SYS_N)

all_sp_cols = sp_6h_mock_h.columns

# co-conditons

In [ ]:
# [co · LFC scatter — Blina vs Mock, NALM-6+T (x) vs Healthy B+T (y), B cells]
# Per-marker log2 fold change (Blina / Mock) for B cells, one panel per timepoint.
# Markers farthest from the y=x diagonal are colored and printed.

def _b_lfc(adata_full, time_val, system_val, layer='raw', pseudo=1.0):
    mask = (
        (adata_full.obs['time'] == time_val) &
        (adata_full.obs['cell_type_annot'] == 'B') &
        (adata_full.obs['cell_system'] == system_val)
    )
    sub = adata_full[mask]
    X = np.array(sub.layers[layer], dtype=np.float32)
    mock  = X[(sub.obs['condition'] == 'Mock').values].mean(axis=0)
    blina = X[(sub.obs['condition'] == 'Blinatumomab').values].mean(axis=0)
    return pd.Series(np.log2((blina + pseudo) / (mock + pseudo)), index=sub.var_names)

SYS_N_CO = 'NALM-6 + healthy T'
SYS_H_CO = 'healthy B + healthy T'
TOP_K = 10  # markers farthest from diagonal to highlight/print

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

for ax, time_val in zip(axes, ['6h', '48h']):
    lfc_n = _b_lfc(adata, time_val, SYS_N_CO)
    lfc_h = _b_lfc(adata, time_val, SYS_H_CO)
    df_lfc = pd.DataFrame({'nalm': lfc_n, 'healthy': lfc_h}).replace(
        [np.inf, -np.inf], np.nan).dropna()

    # Signed perpendicular distance from y=x: positive = above diagonal (higher in healthy)
    df_lfc['off_diag'] = (df_lfc['healthy'] - df_lfc['nalm']) / np.sqrt(2)
    top_idx = df_lfc['off_diag'].abs().nlargest(TOP_K).index
    colors = np.where(df_lfc.loc[top_idx, 'off_diag'] > 0, '#1f77b4', '#d62728')
    color_map = dict(zip(top_idx, colors))

    lim = max(df_lfc[['nalm', 'healthy']].abs().max().max() * 1.15, 0.1)
    ax.axhline(0, color='grey', lw=0.7, ls='--')
    ax.axvline(0, color='grey', lw=0.7, ls='--')
    ax.plot([-lim, lim], [-lim, lim], color='grey', lw=0.7, ls=':')

    other_idx = df_lfc.index.difference(top_idx)
    ax.scatter(df_lfc.loc[other_idx, 'nalm'], df_lfc.loc[other_idx, 'healthy'],
               s=30, alpha=0.55, color='#bbbbbb', edgecolor='white', linewidth=0.5)
    ax.scatter(df_lfc.loc[top_idx, 'nalm'], df_lfc.loc[top_idx, 'healthy'],
               s=70, alpha=0.9, c=[color_map[m] for m in top_idx],
               edgecolor='black', linewidth=0.6, zorder=3)

    for m in top_idx:
        ax.annotate(display_name(m),
                    (df_lfc.loc[m, 'nalm'], df_lfc.loc[m, 'healthy']),
                    fontsize=9, fontweight='bold', color=color_map[m],
                    xytext=(4, 4), textcoords='offset points')

    r = df_lfc['nalm'].corr(df_lfc['healthy'])
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect('equal')
    ax.set_xlabel(f'LFC Blina vs Mock\n{SYS_N_CO}')
    ax.set_ylabel(f'LFC Blina vs Mock\n{SYS_H_CO}')
    ax.set_title(f'B cells  —  {time_val}   (n={len(df_lfc)}, Pearson r={r:.2f})')

    print(f'\n{"=" * 65}')
    print(f'  B cells {time_val}  —  top {TOP_K} markers farthest from y=x')
    print(f'  blue = higher LFC in {SYS_H_CO} | red = higher LFC in {SYS_N_CO}')
    print(f'{"=" * 65}')
    top_tbl = df_lfc.loc[top_idx, ['nalm', 'healthy', 'off_diag']].copy()
    top_tbl.index = [display_name(m) for m in top_tbl.index]
    top_tbl.columns = ['LFC_NALM', 'LFC_Healthy', 'off_diag']
    top_tbl = top_tbl.reindex(top_tbl['off_diag'].abs().sort_values(ascending=False).index)
    print(top_tbl.round(3).to_string())

plt.tight_layout()
plt.show()

In [ ]:
# [co · NALM-6 + healthy T, B cells — Mock vs Blina: abundance + spatial, per timepoint]

SYS_N_CO = 'NALM-6 + healthy T'
TOP_N = 15


def _mw_compare_b(values_a, values_b, labels):
    """Per-feature MW U test. mean_diff = mean(a) - mean(b); positive = higher in a."""
    rows = []
    for j, name in enumerate(labels):
        a, b = values_a[:, j], values_b[:, j]
        mean_diff = a.mean() - b.mean()
        _, p = mannwhitneyu(a, b, alternative='two-sided')
        rows.append({'feature': name, 'mean_diff': mean_diff, 'pval': p})
    df = pd.DataFrame(rows)
    _, df['padj'], _, _ = multipletests(df['pval'], method='fdr_bh')
    return df


def _b_subset(adata_full, time_val, sys_val):
    mask = (
        (adata_full.obs['time'] == time_val) &
        (adata_full.obs['cell_type_annot'] == 'B') &
        (adata_full.obs['cell_system'] == sys_val)
    )
    return adata_full[mask]


def _blina_vs_mock_ab(sub, layer='arcsinh'):
    cond = sub.obs['condition'].values
    X = np.array(sub.layers[layer], dtype=np.float32)
    return _mw_compare_b(X[cond == 'Blinatumomab'], X[cond == 'Mock'], list(sub.var_names))


def _blina_vs_mock_sp(sub, obsm_key='spatial_asinh5'):
    sp = sub.obsm[obsm_key]
    if not isinstance(sp, pd.DataFrame):
        sp = pd.DataFrame(sp, index=sub.obs_names)
    cond = sub.obs['condition'].values
    vals = sp.values
    return _mw_compare_b(vals[cond == 'Blinatumomab'], vals[cond == 'Mock'], list(sp.columns))


def _plot_bars_b(ax, top, color, xlabel, title, label_fmt=display_name, tick_fs=8):
    labels = [label_fmt(f) for f in top['feature']]
    ax.barh(labels, top['mean_diff'], color=color, alpha=0.85)
    ax.axvline(0, color='black', lw=0.7, ls='--')
    ax.set_xlabel(xlabel)
    ax.set_title(title)
    ax.tick_params(axis='y', labelsize=tick_fs)
    xrng = max(top['mean_diff'].abs().max(), 1e-6)
    pad = xrng * 0.02
    for i, (_, r) in enumerate(top.iterrows()):
        s = sig_label(r['padj'])
        xp = r['mean_diff'] + (pad if r['mean_diff'] >= 0 else -pad)
        ha = 'left' if r['mean_diff'] >= 0 else 'right'
        ax.text(xp, i, s, va='center', ha=ha, fontsize=7)


def _pair_label_b(pair_name):
    parts = pair_name.split('/')
    if len(parts) != 2:
        return pair_name
    a, b = parts
    return f'{display_name(a)} / {display_name(b)}'


for time_val in ['6h', '48h']:
    sub = _b_subset(adata, time_val, SYS_N_CO)
    n_mock  = (sub.obs['condition'] == 'Mock').sum()
    n_blina = (sub.obs['condition'] == 'Blinatumomab').sum()
    print(f'\n[{time_val}] B cells in {SYS_N_CO}: Mock n={n_mock}, Blina n={n_blina}')

    da_ab = _blina_vs_mock_ab(sub)
    da_sp = _blina_vs_mock_sp(sub)

    fig, axes = plt.subplots(2, 2, figsize=(16, 13))

    # Row 0 — abundance
    top_up_ab = da_ab.nlargest(TOP_N, 'mean_diff').sort_values('mean_diff')
    top_dn_ab = da_ab.nsmallest(TOP_N, 'mean_diff').sort_values('mean_diff')
    _plot_bars_b(axes[0, 0], top_dn_ab, '#1f77b4',
                 'Mean diff (arcsinh, Blina − Mock)',
                 f'Abundance — top {TOP_N} higher in Mock')
    _plot_bars_b(axes[0, 1], top_up_ab, '#d62728',
                 'Mean diff (arcsinh, Blina − Mock)',
                 f'Abundance — top {TOP_N} higher in Blina')

    # Row 1 — spatial colocalization
    top_up_sp = da_sp.nlargest(TOP_N, 'mean_diff').sort_values('mean_diff')
    top_dn_sp = da_sp.nsmallest(TOP_N, 'mean_diff').sort_values('mean_diff')
    _plot_bars_b(axes[1, 0], top_dn_sp, '#1f77b4',
                 'Mean diff (Blina − Mock)',
                 f'Spatial coloc — top {TOP_N} pairs higher in Mock',
                 label_fmt=_pair_label_b, tick_fs=7)
    _plot_bars_b(axes[1, 1], top_up_sp, '#d62728',
                 'Mean diff (Blina − Mock)',
                 f'Spatial coloc — top {TOP_N} pairs higher in Blina',
                 label_fmt=_pair_label_b, tick_fs=7)

    fig.suptitle(f'B cells — {SYS_N_CO} — {time_val}  (Blina vs Mock)',
                 fontsize=14, y=1.00)
    plt.tight_layout()
    plt.show()

    print(f'\n--- {time_val} abundance (top higher in Blina) ---')
    out = top_up_ab.iloc[::-1][['feature', 'mean_diff', 'padj']].copy()
    out['feature'] = out['feature'].map(display_name)
    print(out.to_string(index=False))
    print(f'\n--- {time_val} spatial coloc (top higher in Blina) ---')
    out_sp = top_up_sp.iloc[::-1][['feature', 'mean_diff', 'padj']].copy()
    out_sp['feature'] = out_sp['feature'].map(_pair_label_b)
    print(out_sp.to_string(index=False))

In [ ]:
# [co · B cells: Healthy B+T vs NALM-6+T — selected proteins across 4 conditions]

TARGET_MARKERS_CO = ['CD274', 'CD152', 'CD47', 'HLA-ABC',
                     'CD37', 'CD35', 'CD32', 'CD180']

SYS_H_CO = 'healthy B + healthy T'
SYS_N_CO = 'NALM-6 + healthy T'

mask_b_co = (
    (adata.obs['cell_type_annot'] == 'B') &
    (adata.obs['cell_system'].isin([SYS_H_CO, SYS_N_CO])) &
    (adata.obs['time'].isin(['6h', '48h'])) &
    (adata.obs['condition'].isin(['Mock', 'Blinatumomab']))
)
adata_co = adata[mask_b_co].copy()
adata_co.obs['time_cond'] = (
    adata_co.obs['time'].astype(str) + ' ' + adata_co.obs['condition'].astype(str)
)
adata_co.obs['system_short'] = adata_co.obs['cell_system'].map({
    SYS_H_CO: 'Healthy',
    SYS_N_CO: 'NALM-6',
}).astype('category')

cond_order = ['6h Mock', '6h Blinatumomab', '48h Mock', '48h Blinatumomab']

present_markers = [m for m in TARGET_MARKERS_CO if m in adata_co.var_names]
missing = [m for m in TARGET_MARKERS_CO if m not in adata_co.var_names]
if missing:
    print(f'Missing markers (skipped): {missing}')
print(f'Plotting {len(present_markers)} markers × {len(cond_order)} conditions × 2 systems')

X_co = pd.DataFrame(
    np.array(adata_co.layers['arcsinh'], dtype=np.float32),
    index=adata_co.obs_names, columns=adata_co.var_names,
)

n_cols = 4
n_rows = int(np.ceil(len(present_markers) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols,
                         figsize=(6 * n_cols, 5 * n_rows), squeeze=False)

for idx, marker in enumerate(present_markers):
    ax = axes[idx // n_cols][idx % n_cols]
    df = pd.DataFrame({
        'expression': X_co[marker].values,
        'time_cond':  adata_co.obs['time_cond'].values,
        'system':     adata_co.obs['system_short'].values,
    })
    sns.violinplot(
        data=df, x='time_cond', y='expression', hue='system',
        order=cond_order, hue_order=['Healthy', 'NALM-6'],
        palette={'Healthy': '#1f77b4', 'NALM-6': '#d62728'},
        split=True, cut=0, inner='quartile', density_norm='width', ax=ax,
    )
    ax.set_xlabel('')
    ax.set_ylabel('arcsinh' if idx % n_cols == 0 else '')
    ax.set_title(display_name(marker), fontsize=11, fontweight='bold')
    ax.tick_params(axis='x', labelsize=8, rotation=20)

    stat_parts = []
    for cond in cond_order:
        v_h = df.loc[(df['time_cond'] == cond) & (df['system'] == 'Healthy'), 'expression'].values
        v_n = df.loc[(df['time_cond'] == cond) & (df['system'] == 'NALM-6'),  'expression'].values
        if len(v_h) > 0 and len(v_n) > 0:
            _, pv = mannwhitneyu(v_h, v_n, alternative='two-sided')
            short = cond.replace(' Blinatumomab', ' Bl').replace(' Mock', ' Mk')
            stat_parts.append(f'{short}: {sig_label(pv)}')
    ax.text(0.5, -0.22, '  |  '.join(stat_parts),
            transform=ax.transAxes, ha='center', fontsize=7)

    if idx == 0:
        ax.legend(fontsize=8, loc='upper right')
    elif ax.get_legend() is not None:
        ax.get_legend().remove()

for k in range(len(present_markers), n_rows * n_cols):
    axes[k // n_cols][k % n_cols].axis('off')

fig.suptitle('B cells — Healthy B+T vs NALM-6+T across 4 conditions',
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

# Summary table
print(f'\n{"="*80}')
print('  Mean arcsinh per system × condition, per marker (MW: Healthy vs NALM-6)')
print(f'{"="*80}')
for marker in present_markers:
    print(f'\n{display_name(marker)}')
    print(f'  {"condition":<20s} {"Healthy":>10s} {"NALM-6":>10s} {"diff(N-H)":>12s} {"Sig":>6s}')
    for cond in cond_order:
        m_cond = (adata_co.obs['time_cond'] == cond)
        v_h = X_co.loc[m_cond & (adata_co.obs['system_short'] == 'Healthy'), marker].values
        v_n = X_co.loc[m_cond & (adata_co.obs['system_short'] == 'NALM-6'),  marker].values
        if len(v_h) == 0 or len(v_n) == 0:
            print(f'  {cond:<20s}   (no data)')
            continue
        _, pv = mannwhitneyu(v_h, v_n, alternative='two-sided')
        print(f'  {cond:<20s} {v_h.mean():>10.3f} {v_n.mean():>10.3f} '
              f'{v_n.mean() - v_h.mean():>12.3f} {sig_label(pv):>6s}')

In [ ]:
'TNFSF9' in adata.var_names

# co-conditons — T/NK markers excluded

Repeat of the LFC scatter and Mock-vs-Blina abundance + spatial analysis above,
but with classical pan-T / TCR / lineage markers and a few NK-lineage receptors
(`CD3e, CD2, CD4, CD5, CD6, CD7, CD8, TCRab, TCRgd, TCRVd2, TCRva7.2, TCRVg9, TCRVB5, CD314, CD159c, NKp80`)
removed so the B cell signal is not drowned. For spatial colocalization, any pair
that contains one of these markers on either side is dropped.

In [ ]:
# [co · LFC scatter (T markers excluded) — Blina vs Mock, NALM-6+T (x) vs Healthy B+T (y), B cells]
# Same as the cell above, but classical pan-T / TCR / lineage markers (and a few
# NK-lineage receptors) are dropped from the scatter and from the top-K ranking,
# so the B cell signal is not drowned by the dominant T/NK features.
# CD274 (PD-L1) is always called out in green, regardless of top-K rank.

T_CELL_EXCLUDE = {
    'CD3e', 'CD2', 'CD4', 'CD5', 'CD6', 'CD7', 'CD8',
    'TCRab', 'TCRgd', 'TCRVd2', 'TCRva7.2', 'TCRVg9', 'TCRVB5',
    'CD314', 'CD159c', 'NKp80',
    'CD279', 'CD337', 'CD69', 'CD134', 'CD16',
    'CD11c', 'CD66b', 'CD163', 'CD1a', 'CD62P', 'CD326', 'CD64',
}

SYS_N_CO = 'NALM-6 + healthy T'
SYS_H_CO = 'healthy B + healthy T'
TOP_K = 10
ALWAYS_HIGHLIGHT = ['CD274']  # PD-L1 — show its location/values regardless of rank
HIGHLIGHT_COLOR = '#2ca02c'

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

for ax, time_val in zip(axes, ['6h', '48h']):
    lfc_n = _b_lfc(adata, time_val, SYS_N_CO)
    lfc_h = _b_lfc(adata, time_val, SYS_H_CO)
    df_lfc = pd.DataFrame({'nalm': lfc_n, 'healthy': lfc_h}).replace(
        [np.inf, -np.inf], np.nan).dropna()

    n_before = len(df_lfc)
    df_lfc = df_lfc.drop(index=[m for m in T_CELL_EXCLUDE if m in df_lfc.index])
    print(f'[{time_val}] dropped {n_before - len(df_lfc)} excluded markers, kept {len(df_lfc)}')

    df_lfc['off_diag'] = (df_lfc['healthy'] - df_lfc['nalm']) / np.sqrt(2)
    top_idx = df_lfc['off_diag'].abs().nlargest(TOP_K).index
    colors = np.where(df_lfc.loc[top_idx, 'off_diag'] > 0, '#1f77b4', '#d62728')
    color_map = dict(zip(top_idx, colors))

    # Force-include the always-highlight markers (e.g. CD274) so they're visible/printed
    extra_idx = [m for m in ALWAYS_HIGHLIGHT
                 if m in df_lfc.index and m not in top_idx]
    for m in extra_idx:
        color_map[m] = HIGHLIGHT_COLOR
    highlight_idx = list(top_idx) + extra_idx

    lim = max(df_lfc[['nalm', 'healthy']].abs().max().max() * 1.15, 0.1)
    ax.axhline(0, color='grey', lw=0.7, ls='--')
    ax.axvline(0, color='grey', lw=0.7, ls='--')
    ax.plot([-lim, lim], [-lim, lim], color='grey', lw=0.7, ls=':')

    other_idx = df_lfc.index.difference(highlight_idx)
    ax.scatter(df_lfc.loc[other_idx, 'nalm'], df_lfc.loc[other_idx, 'healthy'],
               s=30, alpha=0.55, color='#bbbbbb', edgecolor='white', linewidth=0.5)
    ax.scatter(df_lfc.loc[top_idx, 'nalm'], df_lfc.loc[top_idx, 'healthy'],
               s=70, alpha=0.9, c=[color_map[m] for m in top_idx],
               edgecolor='black', linewidth=0.6, zorder=3)
    if extra_idx:
        ax.scatter(df_lfc.loc[extra_idx, 'nalm'], df_lfc.loc[extra_idx, 'healthy'],
                   s=110, alpha=0.95, c=HIGHLIGHT_COLOR, marker='*',
                   edgecolor='black', linewidth=0.8, zorder=4)

    for m in highlight_idx:
        ax.annotate(display_name(m),
                    (df_lfc.loc[m, 'nalm'], df_lfc.loc[m, 'healthy']),
                    fontsize=9, fontweight='bold', color=color_map[m],
                    xytext=(4, 4), textcoords='offset points')

    r = df_lfc['nalm'].corr(df_lfc['healthy'])
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect('equal')
    ax.set_xlabel(f'LFC Blina vs Mock\n{SYS_N_CO}')
    ax.set_ylabel(f'LFC Blina vs Mock\n{SYS_H_CO}')
    ax.set_title(f'B cells  —  {time_val}  (T/NK markers excluded)\n(n={len(df_lfc)}, Pearson r={r:.2f})')

    print(f'\n{"=" * 65}')
    print(f'  B cells {time_val}  —  top {TOP_K} markers farthest from y=x  (T/NK excluded)')
    print(f'  blue = higher LFC in {SYS_H_CO} | red = higher LFC in {SYS_N_CO}')
    print(f'{"=" * 65}')
    top_tbl = df_lfc.loc[top_idx, ['nalm', 'healthy', 'off_diag']].copy()
    top_tbl.index = [display_name(m) for m in top_tbl.index]
    top_tbl.columns = ['LFC_NALM', 'LFC_Healthy', 'off_diag']
    top_tbl = top_tbl.reindex(top_tbl['off_diag'].abs().sort_values(ascending=False).index)
    print(top_tbl.round(3).to_string())

    # Always report CD274 (PD-L1) — show its position and rank vs all kept markers
    print(f'\n  CD274 (PD-L1) — always-on highlight  (green star = forced-shown)')
    rank_full = df_lfc['off_diag'].abs().rank(ascending=False, method='min').astype(int)
    for m in ALWAYS_HIGHLIGHT:
        if m not in df_lfc.index:
            print(f'    {m}: not present in data')
            continue
        row = df_lfc.loc[m]
        in_topk = '✓ in top-K' if m in top_idx else '— outside top-K'
        print(f'    {display_name(m):<18s} '
              f'LFC_NALM={row["nalm"]:+.3f}  LFC_Healthy={row["healthy"]:+.3f}  '
              f'off_diag={row["off_diag"]:+.3f}  '
              f'rank={rank_full[m]}/{len(df_lfc)}  ({in_topk})')

plt.tight_layout()
plt.show()

In [ ]:
# [co · NALM-6 + healthy T, B cells — Mock vs Blina (T/NK markers excluded): abundance + spatial]
# Same analysis as the cell above, but classical pan-T / TCR / lineage markers and
# a few NK-lineage receptors (and any spatial pair touching one) are removed so
# the B cell biology surfaces.

T_CELL_EXCLUDE = {
    'CD3e', 'CD2', 'CD4', 'CD5', 'CD6', 'CD7', 'CD8',
    'TCRab', 'TCRgd', 'TCRVd2', 'TCRva7.2', 'TCRVg9', 'TCRVB5',
    'CD314', 'CD159c', 'NKp80',
    'CD279', 'CD337', 'CD69', 'CD134', 'CD16',
    'CD11c', 'CD66b', 'CD163', 'CD1a', 'CD62P', 'CD326', 'CD64',
}


def _is_t_pair(pair_name, exclude):
    parts = pair_name.split('/')
    return any(p in exclude for p in parts)


SYS_N_CO = 'NALM-6 + healthy T'
TOP_N = 15

for time_val in ['6h', '48h']:
    sub = _b_subset(adata, time_val, SYS_N_CO)
    n_mock  = (sub.obs['condition'] == 'Mock').sum()
    n_blina = (sub.obs['condition'] == 'Blinatumomab').sum()
    print(f'\n[{time_val}] B cells in {SYS_N_CO}: Mock n={n_mock}, Blina n={n_blina}')

    da_ab = _blina_vs_mock_ab(sub)
    da_sp = _blina_vs_mock_sp(sub)

    n_ab_before, n_sp_before = len(da_ab), len(da_sp)
    da_ab = da_ab[~da_ab['feature'].isin(T_CELL_EXCLUDE)].reset_index(drop=True)
    da_sp = da_sp[~da_sp['feature'].apply(lambda c: _is_t_pair(c, T_CELL_EXCLUDE))].reset_index(drop=True)
    print(f'  dropped {n_ab_before - len(da_ab)} abundance markers, '
          f'{n_sp_before - len(da_sp)} spatial pairs (T/NK-marker touching)')

    fig, axes = plt.subplots(2, 2, figsize=(16, 13))

    top_up_ab = da_ab.nlargest(TOP_N, 'mean_diff').sort_values('mean_diff')
    top_dn_ab = da_ab.nsmallest(TOP_N, 'mean_diff').sort_values('mean_diff')
    _plot_bars_b(axes[0, 0], top_dn_ab, '#1f77b4',
                 'Mean diff (arcsinh, Blina − Mock)',
                 f'Abundance — top {TOP_N} higher in Mock')
    _plot_bars_b(axes[0, 1], top_up_ab, '#d62728',
                 'Mean diff (arcsinh, Blina − Mock)',
                 f'Abundance — top {TOP_N} higher in Blina')

    top_up_sp = da_sp.nlargest(TOP_N, 'mean_diff').sort_values('mean_diff')
    top_dn_sp = da_sp.nsmallest(TOP_N, 'mean_diff').sort_values('mean_diff')
    _plot_bars_b(axes[1, 0], top_dn_sp, '#1f77b4',
                 'Mean diff (Blina − Mock)',
                 f'Spatial coloc — top {TOP_N} pairs higher in Mock',
                 label_fmt=_pair_label_b, tick_fs=7)
    _plot_bars_b(axes[1, 1], top_up_sp, '#d62728',
                 'Mean diff (Blina − Mock)',
                 f'Spatial coloc — top {TOP_N} pairs higher in Blina',
                 label_fmt=_pair_label_b, tick_fs=7)

    fig.suptitle(f'B cells — {SYS_N_CO} — {time_val}  (Blina vs Mock, T/NK markers excluded)',
                 fontsize=14, y=1.00)
    plt.tight_layout()
    plt.show()

    print(f'\n--- {time_val} abundance (top higher in Blina, T/NK excluded) ---')
    out = top_up_ab.iloc[::-1][['feature', 'mean_diff', 'padj']].copy()
    out['feature'] = out['feature'].map(display_name)
    print(out.to_string(index=False))
    print(f'\n--- {time_val} spatial coloc (top higher in Blina, T/NK excluded) ---')
    out_sp = top_up_sp.iloc[::-1][['feature', 'mean_diff', 'padj']].copy()
    out_sp['feature'] = out_sp['feature'].map(_pair_label_b)
    print(out_sp.to_string(index=False))

# HLA

In [ ]:
# [HLA · HLA-DR-DP-DQ: 6h Blinatumomab — Healthy vs NALM-6]
if 'HLA-DR-DP-DQ' in adata.var_names:
    print(f"{'='*70}")
    print(f"  HLA-DR-DP-DQ — MHC Class II, Antigen Presentation Hub")
    print(f"  6h Blinatumomab: NALM-6 B cells vs Healthy B cells")
    print(f"{'='*70}")
    print(f"  Function: Combined MHC Class II expression (DR, DP, DQ isotypes);")
    print(f"            central to B cell antigen presentation and CD4+ T cell engagement\n")
    
    run_spatial_comparison('HLA-DR-DP-DQ', sp_6h_blina_n, sp_6h_blina_h,
                           'NALM-6 6h Blina', 'Healthy 6h Blina')
else:
    print("HLA-DR-DP-DQ not found in dataset")

In [ ]:
# [HLA · Colocalization of HLA-DR-DP-DQ with selected inhibitory/modulatory partners]
# Compare across conditions (6h Mock, 6h Blina, 48h Blina) × systems (NALM-6 vs Healthy)

HLA_MARKER = 'HLA-DR-DP-DQ'
PARTNERS = ['CD72', 'CD22', 'CD32', 'CD305', 'CD33']

# Build pair column names (could be either order)
def _find_pair_col(cols, m1, m2):
    """Find the colocalization column for a marker pair."""
    for c in cols:
        parts = c.split('/')
        if len(parts) == 2 and set(parts) == {m1, m2}:
            return c
    return None

# Collect colocalization values across all condition × system combos
sp_dict = {
    ('6h Mock',  'NALM-6'):  sp_6h_mock_n,
    ('6h Mock',  'Healthy'): sp_6h_mock_h,
    ('6h Blina', 'NALM-6'):  sp_6h_blina_n,
    ('6h Blina', 'Healthy'): sp_6h_blina_h,
    ('48h Blina','NALM-6'):  sp_48h_blina_n,
    ('48h Blina','Healthy'): sp_48h_blina_h,
}

rows = []
for partner in PARTNERS:
    pair_col = _find_pair_col(all_sp_cols, HLA_MARKER, partner)
    if pair_col is None:
        print(f"  ⚠ pair {HLA_MARKER}/{partner} not found, skipping")
        continue
    for (cond, system), sp_df in sp_dict.items():
        vals = sp_df[pair_col].values
        for v in vals:
            rows.append({
                'partner': display_name(partner),
                'pair_col': pair_col,
                'condition': cond,
                'system': system,
                'coloc': v,
            })

df_coloc = pd.DataFrame(rows)
cond_order = ['6h Mock', '6h Blina', '48h Blina']

# --- Violin plots: one per partner ---
n_partners = len([p for p in PARTNERS if _find_pair_col(all_sp_cols, HLA_MARKER, p)])
fig, axes = plt.subplots(1, n_partners, figsize=(6 * n_partners, 6), squeeze=False)

for i, partner in enumerate(PARTNERS):
    pair_col = _find_pair_col(all_sp_cols, HLA_MARKER, partner)
    if pair_col is None:
        continue
    ax = axes[0][i]
    sub = df_coloc[df_coloc['partner'] == display_name(partner)]

    sns.violinplot(
        data=sub, x='condition', y='coloc', hue='system',
        order=cond_order, hue_order=['NALM-6', 'Healthy'],
        palette={'NALM-6': '#d62728', 'Healthy': '#1f77b4'},
        split=True, cut=0, inner='quartile', density_norm='width',
        ax=ax,
    )
    ax.set_xlabel('')
    ax.set_ylabel('Colocalization (arcsinh)' if i == 0 else '')
    ax.set_title(f'{display_name(HLA_MARKER)}\n× {display_name(partner)}',
                 fontsize=11, fontweight='bold')
    ax.tick_params(axis='x', labelsize=9, rotation=15)

    # Stats: MW U per condition
    stat_parts = []
    for cond in cond_order:
        n_vals = sub[(sub['condition'] == cond) & (sub['system'] == 'NALM-6')]['coloc'].values
        h_vals = sub[(sub['condition'] == cond) & (sub['system'] == 'Healthy')]['coloc'].values
        if len(n_vals) > 0 and len(h_vals) > 0:
            _, pv = mannwhitneyu(n_vals, h_vals, alternative='two-sided')
            stat_parts.append(f'{cond}: {sig_label(pv)}')
    ax.text(0.5, -0.18, '  |  '.join(stat_parts),
            transform=ax.transAxes, ha='center', fontsize=8)

    if i == 0:
        ax.legend(fontsize=8, loc='upper right')
    elif ax.get_legend() is not None:
        ax.get_legend().remove()

fig.suptitle(f'{display_name(HLA_MARKER)} colocalization with selected partners\nNALM-6 vs Healthy B cells across conditions',
             fontsize=13, fontweight='bold', y=1.04)
plt.tight_layout()
plt.show()

# --- Print summary table ---
print(f"\n{'='*80}")
print(f"  {display_name(HLA_MARKER)} colocalization — NALM-6 vs Healthy B cells")
print(f"{'='*80}")

for partner in PARTNERS:
    pair_col = _find_pair_col(all_sp_cols, HLA_MARKER, partner)
    if pair_col is None:
        continue
    print(f"\n  {display_name(HLA_MARKER)} × {display_name(partner)}")
    print(f"  {'-'*60}")
    print(f"  {'Condition':<14s} {'NALM-6':>10s} {'Healthy':>10s} {'Diff':>10s} {'Sig':>6s}")
    for cond in cond_order:
        n_vals = sp_dict[(cond, 'NALM-6')][pair_col].values
        h_vals = sp_dict[(cond, 'Healthy')][pair_col].values
        mn = n_vals.mean()
        mh = h_vals.mean()
        _, pv = mannwhitneyu(n_vals, h_vals, alternative='two-sided')
        print(f"  {cond:<14s} {mn:>10.3f} {mh:>10.3f} {mn - mh:>10.3f} {sig_label(pv):>6s}")

<cell_type>markdown</cell_type>
## B Cell APC Synapse — Full Analysis

Antigen presentation complex markers on B cells: MHC Class II, costimulatory/inhibitory ligands, adhesion molecules, and B cell co-receptors that form the immunological synapse with T cells.

In [ ]:
# [APC · Define B cell APC synapse markers]
import matplotlib.patches as mpatches
from nalm_utils import draw_force_net

APC_SYNAPSE_CATEGORIES = {
    'MHC Class II (Ag presentation)': (['HLA-DR-DP-DQ', 'HLA-DR', 'HLA-DQ', 'HLA-ABC'], '#e41a1c'),
    'Costimulation':                   (['CD80', 'CD86', 'CD40'],                         '#ff7f00'),
    'Inhibitory / Checkpoint ligands': (['CD274', 'CD273', 'CD32', 'CD72', 'CD305'],      '#377eb8'),
    'B cell co-receptors':             (['CD19', 'CD20', 'CD22', 'CD79a'],                '#4daf4a'),
    'Adhesion (pSMAC)':                (['CD54', 'CD58', 'CD50', 'CD102'],                '#984ea3'),
}

# Filter to markers present in the data
APC_MARKERS = []
node_cmap_apc = {}
for cat, (markers, color) in APC_SYNAPSE_CATEGORIES.items():
    present = [m for m in markers if m in adata.var_names]
    APC_MARKERS.extend(present)
    for m in present:
        node_cmap_apc[m] = color

APC_MARKERS = sorted(set(APC_MARKERS))

print(f'B cell APC synapse marker set ({len(APC_MARKERS)}):')
for cat, (markers, color) in APC_SYNAPSE_CATEGORIES.items():
    present = [m for m in markers if m in adata.var_names]
    missing = [m for m in markers if m not in adata.var_names]
    print(f'  {cat}: {[display_name(m) for m in present]}')
    if missing:
        print(f'    (missing: {missing})')

In [ ]:
# [APC · Abundance: APC synapse markers — 6h Blina NALM-6 vs Healthy]
# Subset to 6h Blinatumomab B cells, both systems
mask_6h_blina_b = (
    (adata.obs['time'] == '6h') &
    (adata.obs['condition'] == 'Blinatumomab') &
    (adata.obs['cell_type_annot'] == 'B') &
    (adata.obs['cell_system'].isin([SYS_A, SYS_B]))
)
adata_6h_blina = adata[mask_6h_blina_b].copy()

X_apc = pd.DataFrame(
    np.array(adata_6h_blina.layers['arcsinh'], dtype=np.float32),
    index=adata_6h_blina.obs_names,
    columns=adata_6h_blina.var_names,
)

nalm_mask = (adata_6h_blina.obs['cell_system'] == SYS_A).values
healthy_mask = (adata_6h_blina.obs['cell_system'] == SYS_B).values

# Violin plots per category
for cat, (markers, color) in APC_SYNAPSE_CATEGORIES.items():
    present = [m for m in markers if m in adata_6h_blina.var_names]
    if not present:
        continue
    
    n_cols = len(present)
    fig, axes = plt.subplots(1, n_cols, figsize=(5 * n_cols, 5), squeeze=False)
    
    for j, marker in enumerate(present):
        ax = axes[0][j]
        n_vals = X_apc.loc[nalm_mask, marker].values
        h_vals = X_apc.loc[healthy_mask, marker].values
        
        df_v = pd.DataFrame({
            'expression': np.concatenate([n_vals, h_vals]),
            'system': ['NALM-6'] * len(n_vals) + ['Healthy'] * len(h_vals),
        })
        
        sns.violinplot(
            data=df_v, x='system', y='expression',
            palette={'NALM-6': '#d62728', 'Healthy': '#1f77b4'},
            cut=0, inner='quartile', ax=ax,
        )
        
        _, pv = mannwhitneyu(n_vals, h_vals, alternative='two-sided')
        ax.set_title(f'{display_name(marker)}\np = {pv:.2e}  {sig_label(pv)}',
                     fontsize=10, fontweight='bold')
        ax.set_xlabel('')
        ax.set_ylabel('arcsinh' if j == 0 else '')
    
    fig.suptitle(f'{cat}\n6h Blinatumomab B cells — NALM-6 vs Healthy',
                 fontsize=12, fontweight='bold', y=1.03)
    plt.tight_layout()
    plt.show()

# Print DA summary
da_apc = compute_da(adata_6h_blina, SYS_A, SYS_B)
da_apc_sub = da_apc[da_apc['marker'].isin(APC_MARKERS)].sort_values('padj')
print(f"\n{'='*70}")
print(f"  APC synapse DA — 6h Blinatumomab: NALM-6 vs Healthy B cells")
print(f"  (positive = higher in NALM-6)")
print(f"{'='*70}")
da_apc_sub_display = da_apc_sub[['marker', 'mean_diff', 'padj']].copy()
da_apc_sub_display['marker'] = da_apc_sub_display['marker'].map(display_name)
print(da_apc_sub_display.to_string(index=False))

In [ ]:
# [APC · Spatial colocalization heatmaps — 6h Blina NALM-6 vs Healthy]
apc_pair_cols = get_pairwise_cols(all_sp_cols, set(APC_MARKERS))
print(f'APC synapse pairwise columns: {len(apc_pair_cols)}')

mat_apc_n = build_mean_matrix(sp_6h_blina_n, apc_pair_cols, APC_MARKERS)
mat_apc_h = build_mean_matrix(sp_6h_blina_h, apc_pair_cols, APC_MARKERS)
mat_apc_diff = mat_apc_n - mat_apc_h

row_linkage_apc = compute_ward_linkage(mat_apc_diff)

# Category legend handles
cat_handles_apc = [mpatches.Patch(color=c, label=cat)
                   for cat, (_, c) in APC_SYNAPSE_CATEGORIES.items()]

# Shared vmax for absolute matrices; separate for diff
vmax_apc = max(mat_apc_n.abs().max().max(), mat_apc_h.abs().max().max())
vmax_diff_apc = mat_apc_diff.abs().max().max()

n_apc = len(APC_MARKERS)
fig_sz = max(10, n_apc * 0.28)
tk_fs = max(5, min(8, 200 // n_apc))
apc_sub = f'B cell APC synapse markers (n={n_apc})'

configs = [
    (mat_apc_n,    'NALM-6 6h Blina',         'RdBu_r',  vmax_apc),
    (mat_apc_h,    'Healthy 6h Blina',         'RdBu_r',  vmax_apc),
    (mat_apc_diff, 'Diff (NALM-6 − Healthy)',  'coolwarm', vmax_diff_apc),
]

# Row/col colors by category
rc_apc = pd.Series({m: node_cmap_apc.get(m, '#555555') for m in APC_MARKERS}, name='Category')

for mat, label, cm, vm in configs:
    dn = {m: display_name(m) for m in mat.index}
    md = mat.rename(index=dn, columns=dn)
    rc_display = pd.Series({display_name(m): node_cmap_apc.get(m, '#555555') for m in mat.index},
                           name='Category')

    g = sns.clustermap(
        md, cmap=cm, center=0, vmin=-vm, vmax=vm,
        row_linkage=row_linkage_apc, col_linkage=row_linkage_apc,
        row_colors=rc_display, col_colors=rc_display,
        figsize=(fig_sz, fig_sz), linewidths=0,
        xticklabels=True, yticklabels=True,
        cbar_kws={'shrink': 0.4, 'label': label},
        dendrogram_ratio=0.08, cbar_pos=(0.02, 0.82, 0.03, 0.15),
    )
    g.ax_heatmap.tick_params(axis='both', labelsize=tk_fs)
    g.ax_heatmap.legend(handles=cat_handles_apc, loc='upper left', fontsize=7,
                        framealpha=0.9, title='Category', title_fontsize=8,
                        bbox_to_anchor=(-0.3, 1.0))
    g.fig.suptitle(f'APC synapse colocalization — {label}\n({apc_sub})', fontsize=12, y=1.01)
    plt.show()

# Print cluster analysis
print(f"\n{'='*70}")
print(f"  APC synapse cluster analysis")
print(f"{'='*70}")
plot_clusters(row_linkage_apc, APC_MARKERS,
              {'NALM-6': mat_apc_n, 'Healthy': mat_apc_h, 'Diff': mat_apc_diff},
              n_clust_list=[3, 5])

In [ ]:
# [APC · Network graphs — 6h Blina NALM-6 vs Healthy]
# Category labels for legend inside draw_force_net
cl_apc = {cat: color for cat, (_, color) in APC_SYNAPSE_CATEGORIES.items()}
for mm in [mat_apc_n, mat_apc_h, mat_apc_diff]:
    mm._node_category_labels = cl_apc

for mat, label in [
    (mat_apc_n,    'NALM-6 6h Blina'),
    (mat_apc_h,    'Healthy 6h Blina'),
    (mat_apc_diff, 'Diff (NALM-6 − Healthy)'),
]:
    fig_net, ax_net = plt.subplots(figsize=(10, 10))
    draw_force_net(mat, f'APC synapse network — {label}\n({apc_sub})',
                   ax=ax_net, top_n=80, highlight_node='HLA-DR-DP-DQ',
                   node_color_map=node_cmap_apc, layout='kamada_kawai')
    plt.tight_layout()
    plt.show()